# MorphIQ SDK for Python — Regime analysis

Examine a ticker's price history and the API's latest regime assessment.

**Run All once to open the workspace.** Yahoo prices load automatically. Enter a ticker
and choose daily, 1-minute, 5-minute, 15-minute, 30-minute or hourly prices, then click
**Analyze regime**. Apply a ticker edit with Enter or by leaving the field.
Changing the ticker or interval refreshes history and clears the previous assessment.
Only **Analyze regime** sends a MorphIQ request (one request, which may consume API credits).

The workspace keeps the latest 512 prices in memory, using the same split-adjusted
Yahoo closes as option pricing, without dividend adjustment. Daily mode uses completed
daily closes; intraday mode uses regular-session bars and shows their timestamps and
timezone. The latest intraday bar may still be forming or delayed. Analysis is disabled
if Yahoo cannot supply 512 valid prices for the selection.

Docker includes everything required. Outside Docker, install this repository with
`pip install -e '.[notebooks]'`. Set `MORPHIQ_API_KEY` before starting, or enter it
at the masked prompt. [Get API access](https://developers.morphiqlabs.com).

[Back to start](../start-here.ipynb)


In [ ]:
import math
import os
from datetime import date, timedelta
from getpass import getpass

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
from IPython.display import display

from morphiq_sdk import APIError, MorphIQ, RegimeRequest, ResponseError, TransportError

if not os.environ.get("MORPHIQ_API_KEY"):
    os.environ["MORPHIQ_API_KEY"] = getpass("MorphIQ API key: ")

ticker = widgets.Text(
    value="SPY",
    description="Ticker",
    continuous_update=False,
    layout=widgets.Layout(width="260px"),
)
price_interval = widgets.Dropdown(
    options=[
        ("Daily close", "1d"),
        ("1 minute", "1m"),
        ("5 minutes", "5m"),
        ("15 minutes", "15m"),
        ("30 minutes", "30m"),
        ("1 hour", "60m"),
    ],
    value="1d",
    description="Price interval",
    layout=widgets.Layout(width="280px"),
    style={"description_width": "initial"},
)
analyze_button = widgets.Button(
    description="Analyze regime · 1 request",
    button_style="primary",
    disabled=True,
    layout=widgets.Layout(width="260px"),
)
history_output = widgets.Output()
analysis_output = widgets.Output()
loaded_closes = None
loaded_ticker = None
loaded_interval = None
last_response = None


def invalidate_history(change):
    global loaded_closes, loaded_ticker, loaded_interval, last_response
    loaded_closes = loaded_ticker = loaded_interval = last_response = None
    analyze_button.disabled = True
    history_output.clear_output()
    analysis_output.clear_output()


ticker.observe(invalidate_history, names="value")
price_interval.observe(invalidate_history, names="value")


def yahoo_history_options(interval):
    if interval == "1d":
        end = date.today()
        return {"interval": interval, "start": end - timedelta(days=3 * 366), "end": end}
    # Fetch enough regular-session bars for 512 prices within Yahoo's lookback limits.
    periods = {"1m": "5d", "5m": "1mo", "15m": "60d", "30m": "60d", "60m": "6mo"}
    return {"interval": interval, "period": periods[interval]}


def price_timestamp(timestamp, interval):
    if interval == "1d":
        return str(timestamp.date())
    return f"{timestamp.isoformat()} ({timestamp.tz or 'timezone not reported'})"


def use_history(history, symbol, interval="1d"):
    global loaded_closes, loaded_ticker, loaded_interval, last_response
    loaded_closes = loaded_ticker = loaded_interval = last_response = None
    analyze_button.disabled = True
    if history is None or history.empty:
        raise ValueError("Yahoo returned no history. Check the ticker.")
    closes = history["Close"].tail(512).copy()
    if len(closes) != 512 or not all(math.isfinite(value) and value > 0 for value in closes):
        raise ValueError(
            f"Need 512 valid positive closes; Yahoo returned {len(closes)} bars in the window. "
            "Try another ticker or interval. "
            "Missing or invalid closes in returned bars are not filled or dropped."
        )
    if not closes.index.is_monotonic_increasing or not closes.index.is_unique:
        raise ValueError("Price dates must be unique and ordered oldest to newest.")
    loaded_closes, loaded_ticker, loaded_interval = closes, symbol, interval
    analyze_button.disabled = False
    print(
        f"{symbol}: {len(closes)} Yahoo {interval} closes, "
        f"{price_timestamp(closes.index[0], interval)} – "
        f"{price_timestamp(closes.index[-1], interval)}"
    )
    print(f"Latest close: {closes.iloc[-1]:.6g}. Split-adjusted, without dividend adjustment.")
    if interval != "1d":
        print("The latest returned bar may still be forming, and Yahoo data may be delayed.")
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(closes.index, closes.values, color="#2459a6")
    ax.set(title=f"{symbol} · {interval} input price history", ylabel="Price")
    ax.grid(alpha=0.25)
    display(fig)
    plt.close(fig)


def load_history(_):
    with history_output:
        history_output.clear_output(wait=True)
        analysis_output.clear_output()
        ticker.disabled = price_interval.disabled = True
        analyze_button.disabled = True
        try:
            symbol = ticker.value.strip().upper()
            if not symbol:
                raise ValueError("Enter a ticker.")
            selected_interval = price_interval.value
            history = yf.download(
                symbol,
                **yahoo_history_options(selected_interval),
                prepost=False,
                ignore_tz=False,
                auto_adjust=False,
                keepna=False,  # Use yfinance's standard empty-bar cleanup.
                multi_level_index=False,
                progress=False,
                threads=False,
            )
            use_history(history, symbol, selected_interval)
        except Exception as error:
            invalidate_history(None)
            print("History lookup failed:", error)
        finally:
            ticker.disabled = price_interval.disabled = False


def analyze_regime(_):
    global last_response
    with analysis_output:
        analysis_output.clear_output(wait=True)
        if (
            loaded_closes is None
            or loaded_ticker != ticker.value.strip().upper()
            or loaded_interval != price_interval.value
        ):
            print("Price history is unavailable for this selection. Change the ticker or interval.")
            return
        ticker.disabled = price_interval.disabled = analyze_button.disabled = True
        last_response = None
        try:
            # Bar labels do not establish the exact final observation time; omit as_of.
            request = RegimeRequest(prices=loaded_closes.tolist())
            with MorphIQ() as client:
                response = client.regime(request)
            last_response = response
            print(
                f"{loaded_ticker} · {loaded_interval} regime at the final supplied close "
                f"({price_timestamp(loaded_closes.index[-1], loaded_interval)})"
            )
            rows = []
            heads = [
                ("Decision", response.result.decision),
                ("Direction", response.result.direction),
                ("State", response.result.state),
            ]
            for name, head in heads:
                rows.append(
                    {
                        "Head": name,
                        "Status": head.status,
                        "Target": head.basis.target,
                        "Result": head.prediction.label
                        if head.status == "available"
                        else head.reason_code,
                    }
                )
            display(pd.DataFrame(rows).set_index("Head"))
            fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
            for ax, (name, head) in zip(axes, heads, strict=True):
                ax.set_title(name)
                if head.status == "available":
                    labels = [score.label for score in head.prediction.scores]
                    scores = [score.value for score in head.prediction.scores]
                    ax.barh(labels, scores, color="#2459a6")
                    ax.set_xlim(0, 1)
                    ax.set_xlabel("Uncalibrated score")
                else:
                    ax.text(
                        0.5,
                        0.5,
                        f"{head.status}\n{head.reason_code}",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                        wrap=True,
                    )
                    ax.set_axis_off()
            fig.tight_layout()
            display(fig)
            plt.close(fig)
            metadata = response.metadata
            consumed = metadata.consumed_price_range
            print(
                f"Consumed {metadata.price_count} of {metadata.received_price_count} "
                "supplied prices: "
                f"indices [{consumed.start_index}, {consumed.end_index_exclusive})."
            )
            print(f"Model: {metadata.provenance.model_id} ({metadata.provenance.model_status})")
            print("Request ID:", metadata.request_id)
            print(
                "CU charged:",
                metadata.fcu_charged if metadata.fcu_charged is not None else "not reported",
            )
            details = {
                "Model SHA-256": metadata.provenance.model_sha256,
                "Preprocessing": metadata.provenance.preprocessing,
                "Reference teacher executed": metadata.provenance.teacher_executed,
                "Return type / unit": f"{metadata.return_type} / {metadata.return_unit}",
                "Selected input index": metadata.input_price_index,
                "Engine(s)": "; ".join(
                    f"{e.name} {e.version} ({e.revision})" for e in metadata.engines
                ),
            }
            display(pd.DataFrame(details.items(), columns=["Provenance", "Value"]))
        except APIError as error:
            charge = error.metadata.fcu_charged if error.metadata else None
            print(f"HTTP {error.status_code}: {error.code}: {error.message}")
            print(
                f"Request ID: {error.request_id}; "
                f"CU charged: {charge if charge is not None else 'not reported'}"
            )
        except (TransportError, ResponseError) as error:
            print("No usable response. Execution and charges may be unknown; no retry was made.")
            if isinstance(error, ResponseError):
                print("Request ID:", error.request_id)
        finally:
            ticker.disabled = price_interval.disabled = analyze_button.disabled = False


analyze_button.on_click(analyze_regime)
display(
    widgets.VBox(
        [
            widgets.HBox(
                [ticker, price_interval, analyze_button],
                layout=widgets.Layout(flex_flow="row wrap"),
            ),
            history_output,
            analysis_output,
        ]
    )
)

In [ ]:
ticker.observe(load_history, names="value")
price_interval.observe(load_history, names="value")
load_history(None)

### Reading the assessment

The assessment is for the final supplied observation. The price chart is input history;
it does not show historical regime classifications. V1 takes the latest 512 prices
and consumes their latest 507; the response reports the exact consumed range.
Changing the interval changes the time span represented by those prices. The API
receives the ordered prices, without a sampling-interval parameter. Daily dates and
intraday bar labels do not establish exact final observation times, so `as_of` is omitted.

Each head has its own status. **Available** includes a prediction; **not applicable**
and **unavailable** include a reason instead. `unresolved` is a valid available decision,
and `NEUTRAL` is a valid direction. `LOWER_VOL` and `HIGHER_VOL` compare the reference
model's two states, not historical volatility norms or price direction.

The bars show **uncalibrated softmax scores**, not calibrated probabilities, p-values
or confidence intervals. Do not multiply scores across heads. The basis describes
what is predicted; it does not imply that reference fitting ran for your request.
Keep the model provenance and request ID with any interpretation.

Missing CU means unreported, not zero. No automatic retries occur. Clear outputs before
sharing and shut down the kernel when finished. Expand the hidden code cell to inspect the implementation.
